# 第24讲：lora_inject

原单元、历史输出及元数据保留；新增说明与代码注释不冒充原作者文字。教学CPU变体见复现记录。


### 原单元 0（markdown）

【建议会】参考低层注入接口。原链接未固定版本；本次固定本机PEFT0.12.0发行文件及SHA，不声称讲师用同一版本。


- references
    - https://huggingface.co/docs/peft/developer_guides/low_level_api

### 原单元 1（code）

【必须会】导入inspect检查继承/源码，torch和nn构造张量/模块，inject_adapter_in_model直接改模型内部结构；导入本身不下载模型。


In [3]:
# 【必须会】导入inspect检查继承/源码，torch和nn构造张量/模块，inject_adapter_in_model直接改模型内部结构；导入本身不下载模型。
# 导入模块命名空间供后续使用。
import inspect
# 导入模块命名空间供后续使用。
import torch
# 从指定包导入本格需要的真实API；导入不等于加载远程权重。
from torch import nn
# 从指定包导入本格需要的真实API；导入不等于加载远程权重。
from peft import inject_adapter_in_model, LoraConfig

[2024-04-04 14:13:19,645] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


### 原单元 2（code）

【必须会】DummyModel继承nn.Module。__init__注册embedding、linear、lm_head；super调用父初始化；forward定义调用流程。输入int64[B,L]→Embedding float32[B,L,1000]→Linear[B,L,1000]→lm_head[B,L,100]。没有激活、softmax或loss，不是训练完的语言模型。


In [4]:
# 【必须会】DummyModel继承nn.Module。__init__注册embedding、linear、lm_head；super调用父初始化；forward定义调用流程。输入int64[B,L]→Embedding float32[B,L,1000]→Linear[B,L,1000]→lm_head[B,L,100]。没有激活、softmax或loss，不是训练完的语言模型。
# 定义nn.Module子类；类是结构模板，实例才持有实际参数。
class DummyModel(torch.nn.Module):
    # 构造方法，实例创建时注册子层。
    def __init__(self):
        # 先初始化父nn.Module内部注册结构。
        super().__init__()
        # 100词映射到1000维；权重[100,1000]。
        self.embedding = torch.nn.Embedding(100, 1000)
        # 目标层权重[1000,1000]、偏置[1000]。
        self.linear = torch.nn.Linear(1000, 1000)
        # 1000维投影为100个未归一化词logit。
        self.lm_head = torch.nn.Linear(1000, 100)

    # 实例被model(input_ids)调用时执行此方法。
    def forward(self, input_ids):
        # int64[B,L]查表得到float32[B,L,1000]。
        x = self.embedding(input_ids)
        # 在末维做线性变换，保留batch/序列轴。
        x = self.linear(x)
        # 末维1000变100，得到[B,L,100]。
        x = self.lm_head(x)
        # 返回最后张量，不额外softmax或计算loss。
        return x

### 原单元 3（code）

【必须会】LoraConfig是配置对象。r64、alpha16令s=alpha/r=.25；dropout .1只在训练分支随机丢输入；bias none不额外训练偏置；仅名字linear匹配，不注入embedding和lm_head。A[64,1000]、B[1000,64]，新增128000参数。


In [5]:
# 【必须会】LoraConfig是配置对象。r64、alpha16令s=alpha/r=.25；dropout .1只在训练分支随机丢输入；bias none不额外训练偏置；仅名字linear匹配，不注入embedding和lm_head。A[64,1000]、B[1000,64]，新增128000参数。
# 创建配置，本行尚未替换模型内部层。
lora_config = LoraConfig(
    # 普通LoRA缩放分子alpha，与r相除得到支路系数。
    lora_alpha=16,
    # 训练时随机丢输入比例；eval时禁用，不能与density混淆。
    lora_dropout=0.1,
    # LoRA中间秩，决定A/B中间轴和新增参数量。
    r=64,
    # none表示不额外训练基础层bias。
    bias="none",
    # 指定注入目标名字，不能只按打印文本猜目标路径。
    target_modules=["linear"],
)

### 原单元 4（code）

【必须会】展示配置，不发生第二次注入。默认普通LoRA且非RSLoRA，不能套alpha/sqrt(r)。


In [6]:
# 【必须会】展示配置，不发生第二次注入。默认普通LoRA且非RSLoRA，不能套alpha/sqrt(r)。
lora_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=None, inference_mode=False, r=64, target_modules={'linear'}, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

### 原单元 5（markdown）

【建议会】先展示未注入的模型，作为后续对象身份/输出对照。


## dummy model

### 原单元 6（code）

【必须会】实例化三个随机层。nn.Module的注册让parameters/state_dict能递归查找子模块；此时没有预训练权重。


In [7]:
# 【必须会】实例化三个随机层。nn.Module的注册让parameters/state_dict能递归查找子模块；此时没有预训练权重。
# 实例化三个随机基础层，不加载预训练权重。
model = DummyModel()
model

DummyModel(
  (embedding): Embedding(100, 1000)
  (linear): Linear(in_features=1000, out_features=1000, bias=True)
  (lm_head): Linear(in_features=1000, out_features=100, bias=True)
)

### 原单元 7（code）

【必须会】type返回注入前的torch.nn.Linear类，不执行前向。


In [8]:
# 【必须会】type返回注入前的torch.nn.Linear类，不执行前向。
# 读取实际Python类，注意torch和PEFT的Linear是不同类。
type(model.linear)

torch.nn.modules.linear.Linear

### 原单元 8（code）

【必须会】LongTensor是int64词ID；[1,8]→[1,8,100]。0–7均在100词表内，最后100维是未归一化logits；本格只看shape。


In [9]:
# 【必须会】LongTensor是int64词ID；[1,8]→[1,8,100]。0–7均在100词表内，最后100维是未归一化logits；本格只看shape。
# 构造有效范围内的int64词ID张量[1,8]。
dummy_inputs = torch.LongTensor([[0, 1, 2, 3, 4, 5, 6, 7]])
# 触发Module.__call__→forward，只查看结果shape。
model(dummy_inputs).shape

torch.Size([1, 8, 100])

### 原单元 9（markdown）

【建议会】进入注入分支。原embedding和lm_head继续使用，linear变成LoRA包装层。


## lora model

### 原单元 10（code）

【必须会】inject_adapter_in_model(config,model)在原模型上替换目标层，返回同一个DummyModel，不产生PeftModel外层。旧linear被保存在base_layer中，基础参数冻结，只有A/B可训练。新分支初始B0，所以初始输出不变。


In [10]:
# 【必须会】inject_adapter_in_model(config,model)在原模型上替换目标层，返回同一个DummyModel，不产生PeftModel外层。旧linear被保存在base_layer中，基础参数冻结，只有A/B可训练。新分支初始B0，所以初始输出不变。
# 在原对象内替换匹配层，返回同一模型实例。
model = inject_adapter_in_model(lora_config, model)
model

DummyModel(
  (embedding): Embedding(100, 1000)
  (linear): lora.Linear(
    (base_layer): Linear(in_features=1000, out_features=1000, bias=True)
    (lora_dropout): ModuleDict(
      (default): Dropout(p=0.1, inplace=False)
    )
    (lora_A): ModuleDict(
      (default): Linear(in_features=1000, out_features=64, bias=False)
    )
    (lora_B): ModuleDict(
      (default): Linear(in_features=64, out_features=1000, bias=False)
    )
    (lora_embedding_A): ParameterDict()
    (lora_embedding_B): ParameterDict()
  )
  (lm_head): Linear(in_features=1000, out_features=100, bias=True)
)

### 原单元 11（code）

【必须会】对比类型：embedding仍是Embedding，linear为PEFT的Linear。类名字相同不代表实现相同，应看完整模块路径。


In [11]:
# 【必须会】对比类型：embedding仍是Embedding，linear为PEFT的Linear。类名字相同不代表实现相同，应看完整模块路径。
# 读取实际Python类，注意torch和PEFT的Linear是不同类。
type(model.embedding), type(model.linear)

(torch.nn.modules.sparse.Embedding, peft.tuners.lora.layer.Linear)

### 原单元 12（code）

【必须会】isinstance检查继承关系；LoRA包装仍然是nn.Module，可以参与注册、调用和autograd。


In [12]:
# 【必须会】isinstance检查继承关系；LoRA包装仍然是nn.Module，可以参与注册、调用和autograd。
# 测试对象是否属于该类或其子类。
isinstance(model.linear, nn.Module)

True

### 原单元 13（code）

【必须会】inspect.getmro返回方法解析顺序元组，解释多继承查找次序，不是模型执行顺序或层数。


In [13]:
# 【必须会】inspect.getmro返回方法解析顺序元组，解释多继承查找次序，不是模型执行顺序或层数。
# 读取类的方法解析顺序，不是神经网络计算顺序。
inspect.getmro(type(model.linear))

(peft.tuners.lora.layer.Linear,
 torch.nn.modules.module.Module,
 peft.tuners.lora.layer.LoraLayer,
 peft.tuners.tuners_utils.BaseTunerLayer,
 abc.ABC,
 object)

### 原单元 14（code）

【必须会】forward??是IPython源码查询。核心为base_layer(x)+B(A(dropout(x)))*s；普通Python不能直接解析双问号，阅读版采用真实IPython转换。本次捕获pinfo2源码文本，不启动外部分页器。


In [14]:
# 【必须会】forward??是IPython源码查询。核心为base_layer(x)+B(A(dropout(x)))*s；普通Python不能直接解析双问号，阅读版采用真实IPython转换。本次捕获pinfo2源码文本，不启动外部分页器。
# result = result + lora_B(lora_A(dropout(x))) * scaling
model.linear.forward??

Signature: model.linear.forward(x: 'torch.Tensor', *args: 'Any', **kwargs: 'Any') -> 'torch.Tensor'
Docstring:
Defines the computation performed at every call.

Should be overridden by all subclasses.

.. note::
    Although the recipe for forward pass needs to be defined within
    this function, one should call the :class:`Module` instance afterwards
    instead of this since the former takes care of running the
    registered hooks while the latter silently ignores them.
Source:   
    def forward(self, x: torch.Tensor, *args: Any, **kwargs: Any) -> torch.Tensor:
        self._check_forward_args(x, *args, **kwargs)
        adapter_names = kwargs.pop("adapter_names", None)

        if self.disable_adapters:
            if self.merged:
                self.unmerge()
            result = self.base_layer(x, *args, **kwargs)
        elif adapter_names is not None:
            result = self._mixed_batch_forward(x, *args, adapter_names=adapter_names, **kwargs)
        elif self.merged:
   

### 原单元 15（markdown）

【建议会】进入保存。只取adapter状态不等于完整模型保存；基础模型、配置和词表仍需另保留。


## save

### 原单元 16（code）

【必须会】get_peft_model_state_dict是函数，按模型的peft_config筛选适配器权重。


In [45]:
# 【必须会】get_peft_model_state_dict是函数，按模型的peft_config筛选适配器权重。
# 从指定包导入本格需要的真实API；导入不等于加载远程权重。
from peft import get_peft_model_state_dict

### 原单元 17（code）

【必须会】返回字典含linear.lora_A.weight[64,1000]和linear.lora_B.weight[1000,64]，默认adapter名在导出键中去掉。没有embedding/base_linear/lm_head完整权重，低层API也不自动提供PeftModel.save_pretrained。


In [48]:
# 【必须会】返回字典含linear.lora_A.weight[64,1000]和linear.lora_B.weight[1000,64]，默认adapter名在导出键中去掉。没有embedding/base_linear/lm_head完整权重，低层API也不自动提供PeftModel.save_pretrained。
# 筛选adapter状态，不包含完整基础模型参数。
get_peft_model_state_dict(model)

{'linear.lora_A.weight': tensor([[-0.0066, -0.0240,  0.0164,  ..., -0.0070,  0.0205,  0.0312],
         [-0.0244, -0.0162, -0.0250,  ...,  0.0038, -0.0161, -0.0171],
         [ 0.0122, -0.0050, -0.0068,  ..., -0.0132,  0.0286,  0.0253],
         ...,
         [-0.0158, -0.0284, -0.0248,  ...,  0.0025,  0.0007,  0.0157],
         [-0.0013, -0.0223, -0.0096,  ..., -0.0131, -0.0036,  0.0033],
         [-0.0202,  0.0143,  0.0211,  ...,  0.0237, -0.0268, -0.0189]]),
 'linear.lora_B.weight': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]])}